# Colab Setup - World-Model Predictive-Control Proof

This notebook prepares the Colab runtime and Google Drive folders for future experiments. It does not train models.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
except Exception as exc:
    print("PyTorch check failed:", repr(exc))

subprocess.run(["nvidia-smi"], check=False)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/wm_poc")

subdirs = [
    "data/dino_wm",
    "data/dmc",
    "data/metaworld",
    "data/robomimic_optional",
    "data/libero_optional",
    "checkpoints/r2dreamer",
    "checkpoints/dino_wm",
    "checkpoints/local_global",
    "logs/r2dreamer",
    "logs/dino_wm",
    "logs/local_global",
    "logs/system",
    "figures/r2dreamer",
    "figures/dino_wm",
    "figures/local_global",
    "tensorboard/r2dreamer",
    "tensorboard/dino_wm",
    "tensorboard/local_global",
    "videos/r2dreamer",
    "videos/dino_wm",
    "videos/local_global",
    "external_repos/r2dreamer",
    "external_repos/dino_wm",
    "external_repos/jepa_wms_optional",
    "reports",
]

for subdir in subdirs:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print(f"Drive tree ready at: {DRIVE_ROOT}")

In [ ]:
import os

os.environ["WM_POC_REPO"] = "/content/World_Models_LAS"
os.environ["WM_POC_DRIVE_ROOT"] = "/content/drive/MyDrive/wm_poc"
os.environ["WM_POC_DATA_DIR"] = "/content/drive/MyDrive/wm_poc/data"
os.environ["WM_POC_CKPT_DIR"] = "/content/drive/MyDrive/wm_poc/checkpoints"
os.environ["WM_POC_LOG_DIR"] = "/content/drive/MyDrive/wm_poc/logs"
os.environ["WM_POC_FIG_DIR"] = "/content/drive/MyDrive/wm_poc/figures"
os.environ["WM_POC_FIGURE_DIR"] = "/content/drive/MyDrive/wm_poc/figures"
os.environ["WM_POC_TB_DIR"] = "/content/drive/MyDrive/wm_poc/tensorboard"
os.environ["WM_POC_VIDEO_DIR"] = "/content/drive/MyDrive/wm_poc/videos"
os.environ["WM_POC_EXTERNAL_REPOS"] = "/content/external_repos"
os.environ["WM_POC_REPORT_DIR"] = "/content/drive/MyDrive/wm_poc/reports"

for key in [
    "WM_POC_REPO",
    "WM_POC_DRIVE_ROOT",
    "WM_POC_DATA_DIR",
    "WM_POC_CKPT_DIR",
    "WM_POC_LOG_DIR",
    "WM_POC_EXTERNAL_REPOS",
]:
    print(key, "=", os.environ[key])

If this notebook is opened directly from GitHub, the repository may not be cloned into `/content`. Use the cell below, or replace the URL if using a fork.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Thomas-Georges/World_Models_LAS.git"
REPO_DIR = Path("/content/World_Models_LAS")


def run_git(args, *, check=True):
    print("$", " ".join(str(arg) for arg in args))
    return subprocess.run(args, check=check)


if (REPO_DIR / ".git").is_dir():
    print(f"Repository already exists at {REPO_DIR}")
    run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    print(f"Cloning {REPO_URL} into {REPO_DIR}")
    run_git(["git", "clone", REPO_URL, str(REPO_DIR)])

run_git(["git", "-C", str(REPO_DIR), "status", "--short"])

In [ ]:
%%bash
set -e
cd /content/World_Models_LAS

python scripts/verify_environment.py --cpu-only
python scripts/verify_drive_layout.py --drive-root /content/drive/MyDrive/wm_poc

The generic external repository clone step is optional. For the R2-Dreamer experiment, prefer the dedicated setup in `01_r2dreamer_foundation.ipynb`.

In [ ]:
%%bash
set -e

cd /content/World_Models_LAS
mkdir -p /content/external_repos

bash scripts/clone_external_repos.sh   --external-root /content/external_repos

At this point the foundation is ready. The next phase is to install dependencies for the selected external repo and run a small smoke test. Do not start long training until the smoke test passes.